In [145]:
import pandas as pd
import re
import ast

In [146]:
# load the oscars and movies dataset
oscars = pd.read_csv("the_oscar_award.csv")
df_full = pd.read_csv("movies_dataset.csv")



In [147]:
# aggregate Oscar data to the film-year level.
oscars_film = (
    oscars
    .groupby(["film", "year_film"])
    .agg(
        oscar_nominations=("category", "count"),
        oscar_wins=("winner", "sum")
    )
    .reset_index()
)

In [148]:
oscars_film["oscar_nominated"] = oscars_film["oscar_nominations"] > 0
oscars_film["oscar_won"] = oscars_film["oscar_wins"] > 0

In [149]:
# standardize movie titles for merging.
def clean_title(title):
    title = str(title).lower()
    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"[^a-z0-9 ]", "", title)
    title = title.strip()
    return title

In [150]:
df_full["title_clean"] = df_full["title"].apply(clean_title)
oscars_film["title_clean"] = oscars_film["film"].apply(clean_title)

In [151]:
df_full["release_year"] = pd.to_datetime(
    df_full["release_date"], errors="coerce"
).dt.year

In [152]:
df_merged = df_full.merge(
    oscars_film,
    left_on=["title_clean", "release_year"],
    right_on=["title_clean", "year_film"],
    how="left"
)

In [153]:
# fill missing Oscar information for unmatched movies.
df_merged["oscar_nominations"] = df_merged["oscar_nominations"].fillna(0)
df_merged["oscar_wins"] = df_merged["oscar_wins"].fillna(0)

df_merged["oscar_nominated"] = df_merged["oscar_nominated"].fillna(False)
df_merged["oscar_won"] = df_merged["oscar_won"].fillna(False)

In [154]:
df_merged["oscar_nominated"].mean()

np.float64(0.0413447782546495)

About 4.1% of the movies in the merged dataset received at least one Academy Award nomination.

In [155]:
# convert stored genre strings into python lists.
df_merged["genre_list"] = df_merged["genres"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [156]:
keep_columns = [
    "id",
    "title",
    "release_year",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "genre_list",
    "popularity",
    "oscar_nominations",
    "oscar_wins",
    "oscar_nominated",
    "oscar_won"
]

In [157]:
df_merged = df_merged[keep_columns].copy()

In [158]:
df_merged = df_merged[
    (df_merged["budget"] > 0) 
]

In [159]:
df_merged.to_csv("oscars_movies_merged.csv", index=False)
